In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Smart mcq solver**

## milestone-0

In [2]:
%pip install -q transformers datasets accelerate scikit-learn tqdm sentencepiece protobuf gensim ## Install required libraries


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [3]:
# Import all necessary packages

import os
import re
import glob
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pandas as pd
import numpy as np

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMultipleChoice,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore") # hide warning messages for cleaner output


import sentencepiece #needed for some transformer models (DeBERTa uses SentencePiece tokenization)

os.environ["TOKENIZERS_PARALLELISM"] = "false" #disable tokenizer parallelism warnings

In [4]:
 # Set random seeds for reproducibility

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Check what computing device is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device name ", device)


device name  cuda


In [5]:
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

DEBERTA_MODEL_PATH = "microsoft/deberta-v3-base"
DISTILBERT_MODEL_PATH = "distilbert-base-uncased"

SCRATCH_MODEL_WEIGHTS = "best_model_scratch.pt" ## saved model weights 


# training flags
RUN_TRAINING_FROM_SCRATCH = False
RUN_TRANSFORMER_TRAINING = True

# ANN hyperparameters
ANN_EPOCHS = 5
ANN_BATCH_SIZE = 32
ANN_LR = 0.001
ANN_EMBED_DIM = 256
ANN_HIDDEN_DIM = 256

# Transformer Hyperparameters 
TRANSFORMER_EPOCHS = 10
TRANSFORMER_BATCH_SIZE = 2
DEBERTA_LR = 5e-6
DISTILBERT_LR = 2e-5
GRAD_ACCUM_STEPS = 8

# ensemble threshold 
MIN_USABLE_MAP3 = 0.40

LABEL_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}



In [6]:
# Load the datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Basic dataset information
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

# Find the answer column
LABEL_COL = None

for col in ["answer", "Answer", "label", "Label", "correct_answer"]:
    if col in train_df.columns:
        LABEL_COL = col
        break

if LABEL_COL:
    print("Answer column:", LABEL_COL)
    print(train_df[LABEL_COL].value_counts())
else:
    print("No answer column found.")
    RUN_TRAINING_FROM_SCRATCH = False
    RUN_TRANSFORMER_TRAINING = False

Train shape: (2000, 8)
Test shape: (500, 7)
Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
Answer column: answer
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [7]:
train_df.head(2)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


In [8]:
# Remove duplicate questions
train_df_clean = train_df.drop_duplicates(
    subset=["prompt", "A", "B", "C", "D", "E"]
)

print("Original dataset size:", len(train_df))
print("After removing duplicates:", len(train_df_clean))
print("Duplicates removed:", len(train_df) - len(train_df_clean))

Original dataset size: 2000
After removing duplicates: 1817
Duplicates removed: 183


In [9]:
# split the data into training and validation sets
gss = GroupShuffleSplit(test_size=0.1, n_splits=1, random_state=42)

train_idx, val_idx = next(
    gss.split(train_df, groups=train_df["prompt"])
)

t_df = train_df.iloc[train_idx]
v_df = train_df.iloc[val_idx]

print("Training samples:", len(t_df))
print("Validation samples:", len(v_df))


Training samples: 1806
Validation samples: 194


## Milestone 1 — (NLP Foundation & Semantic Similarity)

In [10]:
# Clean text by converting to lowercase and removing special characters
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [11]:
# Check for missing data
print(train_df.isnull().sum())
print(test_df.isnull().sum())

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64


In [12]:
# Calculate mAP@3 score
def calculate_map3(logits, targets):
    top3 = np.argsort(logits, axis=1)[:, ::-1][:, :3]

    scores = []
    for pred, true in zip(top3, targets):
        if true == pred[0]:
            scores.append(1.0)
        elif true == pred[1]:
            scores.append(0.5)
        elif true == pred[2]:
            scores.append(1 / 3)
        else:
            scores.append(0.0)

    return np.mean(scores)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build TF-IDF vocabulary
all_text = []
for col in ["prompt", "A", "B", "C", "D", "E"]:
    all_text.extend(train_df[col].astype(str).apply(clean_text))

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)
tfidf_vectorizer.fit(all_text)

print("TF-IDF vocabulary size:", len(tfidf_vectorizer.vocabulary_))

# Predict answers using cosine similarity
def tfidf_predict(df, vectorizer):
    predictions = []

    for _, row in df.iterrows():
        prompt_vec = vectorizer.transform([clean_text(row["prompt"])])

        scores = []
        for option in ["A", "B", "C", "D", "E"]:
            option_vec = vectorizer.transform([clean_text(row[option])])
            scores.append(cosine_similarity(prompt_vec, option_vec)[0][0])

        predictions.append(scores)

    return np.array(predictions)

tfidf_preds = tfidf_predict(v_df, tfidf_vectorizer)
tfidf_targets = v_df[LABEL_COL].map(LABEL_MAP).values

tfidf_map3 = calculate_map3(tfidf_preds, tfidf_targets)

print("TF-IDF mAP@3:", round(tfidf_map3, 4))

TF-IDF vocabulary size: 2761
TF-IDF mAP@3: 0.25


In [14]:
from gensim.models import Word2Vec

# Prepare text for Word2Vec
sentences = []

for col in ["prompt", "A", "B", "C", "D", "E"]:
    for text in train_df[col].astype(str):
        tokens = clean_text(text).split()
        if tokens:
            sentences.append(tokens)

# Train the Word2Vec model
w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10,
    seed=42
)

print("Vocabulary size:", len(w2v_model.wv))

# Convert text into a vector
def text_to_w2v_vector(text, model):
    tokens = clean_text(text).split()
    vectors = [model.wv[token] for token in tokens if token in model.wv]

    if vectors:
        return np.mean(vectors, axis=0)

    return np.zeros(model.vector_size)

# Predict answers using cosine similarity
def w2v_predict(df, model):
    predictions = []

    for _, row in df.iterrows():
        prompt_vec = text_to_w2v_vector(row["prompt"], model).reshape(1, -1)

        scores = []
        for option in ["A", "B", "C", "D", "E"]:
            option_vec = text_to_w2v_vector(row[option], model).reshape(1, -1)
            scores.append(cosine_similarity(prompt_vec, option_vec)[0][0])

        predictions.append(scores)

    return np.array(predictions)

w2v_preds = w2v_predict(v_df, w2v_model)
w2v_targets = v_df[LABEL_COL].map(LABEL_MAP).values

w2v_map3 = calculate_map3(w2v_preds, w2v_targets)

print("Word2Vec mAP@3:", round(w2v_map3, 4))

Vocabulary size: 2973
Word2Vec mAP@3: 0.3376


## Milestone 2 (Enter the Transformers)

In [15]:
# Load the DeBERTa tokenizer
tokenizer_demo = AutoTokenizer.from_pretrained(DEBERTA_MODEL_PATH)

# Tokenize a sample question
sample_text = train_df["prompt"].iloc[0]

simple_tokens = clean_text(sample_text).split()
deberta_tokens = tokenizer_demo.tokenize(sample_text)

print("Simple tokens:", simple_tokens[:10])
print("DeBERTa tokens:", deberta_tokens[:10])

# Convert tokens to IDs
token_ids = tokenizer_demo.convert_tokens_to_ids(deberta_tokens[:10])
print("Token IDs:", token_ids)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Simple tokens: ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heidegger', 's']
DeBERTa tokens: ['▁Pick', '▁the', '▁best', '▁possible', '▁answer', ':', '▁What', '▁is', '▁Martin', '▁Heidegger']
Token IDs: [8498, 262, 410, 628, 1330, 294, 458, 269, 2963, 49965]


In [16]:
# Load the pre-trained model
model_demo = AutoModel.from_pretrained(DEBERTA_MODEL_PATH).to(device)
model_demo.eval()

def get_transformer_embedding(text, tokenizer, model, max_len=128):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

sample_row = train_df.iloc[0]

prompt_emb = get_transformer_embedding(
    str(sample_row["prompt"])[:200],
    tokenizer_demo,
    model_demo
)

print("Embedding shape:", prompt_emb.shape)

for col in ["A", "B", "C", "D", "E"]:
    opt_emb = get_transformer_embedding(
        str(sample_row[col])[:200],
        tokenizer_demo,
        model_demo
    )

    sim = cosine_similarity(prompt_emb, opt_emb)[0][0]
    print(f"{col}: {sim:.4f}")

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (1, 768)
A: 0.3673
B: 0.5305
C: 0.9250
D: 0.3770
E: 0.8692


In [17]:
# Zero-shot prediction using transformer embeddings
def transformer_embedding_predict(df, tokenizer, model, max_len=128):
    predictions = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Zero-shot predictions"):
        prompt_emb = get_transformer_embedding(
            str(row["prompt"])[:300],
            tokenizer,
            model,
            max_len
        )

        option_sims = []
        for col in ["A", "B", "C", "D", "E"]:
            opt_emb = get_transformer_embedding(
                str(row[col])[:300],
                tokenizer,
                model,
                max_len
            )

            sim = cosine_similarity(prompt_emb, opt_emb)[0][0]
            option_sims.append(sim)

        predictions.append(option_sims)

    return np.array(predictions)

zs_preds = transformer_embedding_predict(v_df, tokenizer_demo, model_demo)
zs_targets = v_df[LABEL_COL].map(LABEL_MAP).values

zs_map3 = calculate_map3(zs_preds, zs_targets)

print("Zero-shot mAP@3:", round(zs_map3, 4))

del model_demo

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Zero-shot predictions:   0%|          | 0/194 [00:00<?, ?it/s]

Zero-shot mAP@3: 0.4218


## Milestone 3 (Context Augmentation with RAG Pipelines)

In [18]:
# Build the vector database
rag_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

train_prompt_vectors = rag_vectorizer.fit_transform(
    t_df["prompt"].astype(str).apply(clean_text)
)

print("Training questions:", train_prompt_vectors.shape[0])
print("Feature size:", train_prompt_vectors.shape[1])

Training questions: 1806
Feature size: 796


In [19]:
# Find the most similar questions from the training data

def retrieve_similar(query, vectorizer, vectors, source_df, top_k=3):
    
    query_vec = vectorizer.transform([clean_text(query)]) # Convert the query into a TF-IDF vector

    # Calculate cosine similarity with all training questions
    similarities = cosine_similarity(query_vec, vectors)[0]     # Calculate cosine similarity with all training questions


    top_indices = similarities.argsort()[-top_k:][::-1]     # Get the indices of the top matching questions

    results = []
    for idx in top_indices:
        results.append({
            "prompt": source_df.iloc[idx]["prompt"],
            "answer": source_df.iloc[idx][LABEL_COL],
            "similarity": similarities[idx]
        })

    return results

sample_query = str(test_df["prompt"].iloc[0])# Retrieve similar questions for one test example

similar_questions = retrieve_similar(
    sample_query,
    rag_vectorizer,
    train_prompt_vectors,
    t_df
)

print("Sample query:")
print(sample_query[:150] + "...")

print("Most similar questions:")
for i, s in enumerate(similar_questions, 1):
    print(f"{i}. Similarity: {s['similarity']:.3f}")
    print("Ans:", s["answer"])
    print(s["prompt"][:120] + "...")

Sample query:
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully....
Most similar questions:
1. Similarity: 1.000
Ans: A
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...
2. Similarity: 1.000
Ans: A
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...
3. Similarity: 0.979
Ans: A
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...


In [20]:
# Retrieve similar questions for a sample test question
sample_row = test_df.iloc[0]
sample_prompt = str(sample_row["prompt"])

retrieved = retrieve_similar(
    sample_prompt,
    rag_vectorizer,
    train_prompt_vectors,
    t_df,
    top_k=2
)


augmented_prompt = "Retrieved Context" # create an augmented prompt by adding retrieved context

for i, r in enumerate(retrieved, 1):
    augmented_prompt += (
        f"Similar Question {i} (Answer: {r['answer']}): "
        f"{r['prompt'][:100]}..."
    )

augmented_prompt += f"Current Question{sample_prompt[:200]}"

print(augmented_prompt)

Retrieved ContextSimilar Question 1 (Answer: A): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in ...Similar Question 2 (Answer: A): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in ...Current QuestionPick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.


In [21]:
# add retrieved context to each question
def augment_dataframe(df, is_train=False):
    augmented_prompts = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting prompts"):
        query = str(row["prompt"])

        top_k = 2 if is_train else 1
        retrieved = retrieve_similar(
            query,
            rag_vectorizer,
            train_prompt_vectors,
            t_df,
            top_k=top_k
        )

        # Skip the same question when creating the training set
        if is_train and len(retrieved) > 1 and retrieved[0]["prompt"] == query:
            context_q = retrieved[1]
        else:
            context_q = retrieved[0]

        aug_prompt = (
            f"Context: {context_q['prompt'][:100]}... "
            f"Answer: {context_q['answer']}. "
            f"Question: {query}"
        )

        augmented_prompts.append(aug_prompt)

    df_aug = df.copy()
    df_aug["prompt"] = augmented_prompts

    return df_aug

# Update the datasets with augmented prompts
t_df = augment_dataframe(t_df, is_train=True)
v_df = augment_dataframe(v_df)
test_df = augment_dataframe(test_df)

Augmenting prompts:   0%|          | 0/1806 [00:00<?, ?it/s]

Augmenting prompts:   0%|          | 0/194 [00:00<?, ?it/s]

Augmenting prompts:   0%|          | 0/500 [00:00<?, ?it/s]